# SurvFace — 02. 106-point landmark region materialization

정렬된 SurvFace crop 전체에 106-point landmark를 생성하고 얼굴·눈·코·입·볼·턱 영역 lineage를 고정합니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "survface"

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")

from research.experiments import materialize_step4_landmark_regions
from research.runtime import ProgressReporter

PROGRESS = ProgressReporter(
    "SurvFace 106-point landmarks",
    heartbeat_seconds=None,
    milestone_percent=10,
)


In [2]:
if EXECUTE_STAGE:
    result = materialize_step4_landmark_regions(
        CONFIG_PATH,
        project_root=PROJECT_ROOT,
        dataset_id=DATASET_ID,
        execution_acknowledged=True,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:"),
    )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'has_user_compute_stream': '0', 'gpu_external_alloc': '0', 'enable_cuda_graph': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0'}, 'CPUExecutionProvider': {}}
[00:06:09] SurvFace 106-point landmarks | 106-point landmark materialization | elapsed=3m 54s | progress=10% processed=46335 total=463341 rate=198.04/s eta=35m 06s
[00:10:08] SurvFace 106-point landmarks | 106-point landmark materialization | elapsed=7m 54s | progress=20% processed=92669 total=463341 rate=195.59/s eta=31m 3

{'dataset_id': 'survface',
 'landmark_region_bundle_dir': 'C:\\ronbun\\data\\interim\\step4\\survface\\landmark_regions_106',
 'complete': True}

## 다음 단계

다음은 `survface/04_gradcam/prerequisite/00_source_and_model_freeze.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.